In [1]:
import pandas as pd
import os

# --- Load the raw data ---
raw_data_path = '../data/raw/who_measles_cases.csv'
df = pd.read_csv(raw_data_path)

# --- Initial Inspection ---
print("--- Data Info ---")
df.info()

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Missing Values ---")
print(df.isnull().sum())

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9049 entries, 0 to 9048
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Id                  9049 non-null   int64  
 1   IndicatorCode       9049 non-null   object 
 2   SpatialDimType      9049 non-null   object 
 3   SpatialDim          9049 non-null   object 
 4   ParentLocationCode  8434 non-null   object 
 5   TimeDimType         9049 non-null   object 
 6   ParentLocation      8434 non-null   object 
 7   Dim1Type            0 non-null      float64
 8   Dim1                0 non-null      float64
 9   TimeDim             9049 non-null   int64  
 10  Dim2Type            0 non-null      float64
 11  Dim2                0 non-null      float64
 12  Dim3Type            0 non-null      float64
 13  Dim3                0 non-null      float64
 14  DataSourceDimType   0 non-null      float64
 15  DataSourceDim       0 non-null      f

In [2]:
# --- 1. Select only the columns we need ---
# We only care about the country, year, and case count.
df_clean = df[['SpatialDim', 'TimeDim', 'NumericValue']].copy()

# --- 2. Rename columns for clarity ---
df_clean.rename(columns={
    'SpatialDim': 'CountryCode',
    'TimeDim': 'Year',
    'NumericValue': 'Cases'
}, inplace=True)

print("--- DataFrame after renaming and selecting columns ---")
print(df_clean.head())


# --- 3. Convert 'Year' to a proper datetime format ---
# This is crucial for any time-series analysis.
df_clean['Year'] = pd.to_datetime(df_clean['Year'], format='%Y')

print("\n--- Data types after converting Year ---")
df_clean.info()


# --- 4. Reshape the data from 'long' to 'wide' format ---
# We want Years as our index and each country as a separate column.
# This structure is ideal for comparing and modeling time series.
df_pivot = df_clean.pivot_table(index='Year', columns='CountryCode', values='Cases')

print("\n--- Reshaped (Pivoted) DataFrame ---")
print(df_pivot.head())


# --- 5. Handle missing values in the pivoted data ---
# Pivoting often creates NaNs (e.g., a country didn't report data for a certain year).
# For now, we will fill them with 0, assuming no report means no cases.
# A more advanced method would be interpolation, which we can explore later.
df_pivot.fillna(0, inplace=True)

print("\n--- Pivoted DataFrame after filling missing values ---")
print(df_pivot.head())


# --- 6. Save the processed data ---
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)
processed_file_path = os.path.join(processed_dir, 'measles_cases_processed_timeseries.csv')

df_pivot.to_csv(processed_file_path)

print(f"\nProcessed data successfully saved to: {processed_file_path}")

--- DataFrame after renaming and selecting columns ---
  CountryCode  Year   Cases
0         TGO  1978     0.0
1         PYF  1999     0.0
2         CZE  1992     0.0
3         LKA  1986     3.0
4      WB_LMI  2013  3518.0

--- Data types after converting Year ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9049 entries, 0 to 9048
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   CountryCode  9049 non-null   object        
 1   Year         9049 non-null   datetime64[ns]
 2   Cases        9049 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 212.2+ KB

--- Reshaped (Pivoted) DataFrame ---
CountryCode  ABW     AFG  AFR    AGO  AIA  ALB  AMR  AND  ARE    ARG  ...  \
Year                                                                  ...   
1974-01-01   NaN     NaN  NaN   27.0  NaN  NaN  NaN  NaN  NaN  290.0  ...   
1975-01-01   NaN    22.0  NaN    7.0  NaN 